# Re-export verified Stage 00–02 artifacts

Attach the completed Stage 00, 01, and 02 datasets and run on CPU. Publish the four files in each provenance dataset folder. Use only the revised Stage 00 and Stage 02 releases in Notebook 03.

In [1]:
import hashlib
import json
import re
from pathlib import PurePosixPath


ARTIFACT_SCHEMA = "openapi-artifact-v1"
PRODUCER_NOTEBOOKS = {
    "00": "00_protocol_and_runtime",
    "01": "01_corpus_preparation_and_eligibility",
    "02": "02_operation_slicing_and_prose_measurement",
    "03": "03_pair_and_evaluator_feasibility",
    "04": "04_strata_and_pilot_selection",
    "05": "05_oracle_build_and_review",
    "06": "06_build_corrected_evaluator_and_generation_tasks",
    "07": "07_pilot_generation",
    "08": "08_pilot_execution_and_review",
}


def provenance_digest(value):
    data = json.dumps(value, sort_keys=True, separators=(",", ":"),
                      ensure_ascii=False, allow_nan=False).encode("utf-8")
    return hashlib.sha256(data).hexdigest()


def artifact_path_for(record):
    paths = {record[key] for key in ("artifact_path", "relative_path", "path")
             if key in record}
    if len(paths) != 1:
        raise RuntimeError("Missing or inconsistent artifact paths")
    name = paths.pop()
    path = PurePosixPath(name)
    if not name or name == "." or path.is_absolute() or ".." in path.parts or "\\" in name or path.as_posix() != name:
        raise RuntimeError(f"Invalid artifact path: {name!r}")
    return name


def manifest_digest_valid(value):
    return isinstance(value, str) and re.fullmatch(r"[0-9a-f]{64}", value) is not None


def register_artifact_provenance(manifest):
    stage = manifest["stage"]
    artifacts = manifest["artifacts"]
    if "upstream_manifest_sha256" not in manifest:
        if stage != "00":
            raise RuntimeError(f"Stage {stage} has no upstream identity")
        manifest["upstream_manifest_sha256"] = {}
    links = manifest["upstream_manifest_sha256"]
    config_hash = manifest.get("configuration_hash")
    if config_hash is None:
        config_hash = provenance_digest({artifact_path_for(row): row["sha256"]
                                         for row in artifacts
                                         if artifact_path_for(row).startswith("config/")})
    manifest["configuration_hash"] = config_hash
    manifest["artifact_record_schema"] = ARTIFACT_SCHEMA
    upstream_hash = provenance_digest(links)
    for row in artifacts:
        row.update(artifact_path=artifact_path_for(row), schema_version=ARTIFACT_SCHEMA,
                   producer_notebook=PRODUCER_NOTEBOOKS[stage], config_hash=config_hash,
                   upstream_manifest_hash=upstream_hash)
    verify_artifact_provenance(manifest)


def verify_artifact_provenance(manifest):
    stage = manifest["stage"]
    if manifest.get("artifact_record_schema") != ARTIFACT_SCHEMA:
        raise RuntimeError(f"Stage {stage} requires a corrected provenance export")
    links = manifest["upstream_manifest_sha256"]
    config_hash = manifest["configuration_hash"]
    if (not isinstance(links, dict) or bool(links) != (stage != "00")
            or any(key not in PRODUCER_NOTEBOOKS or key >= stage or not manifest_digest_valid(value)
                   for key, value in links.items()) or not manifest_digest_valid(config_hash)):
        raise RuntimeError(f"Invalid Stage {stage} provenance")
    expected = (ARTIFACT_SCHEMA, PRODUCER_NOTEBOOKS[stage], config_hash, provenance_digest(links))
    seen = set()
    artifacts = manifest["artifacts"]
    if not artifacts or len(artifacts) != manifest["artifact_count"]:
        raise RuntimeError(f"Invalid Stage {stage} artifact count")
    for row in artifacts:
        name = artifact_path_for(row)
        actual = tuple(row.get(key) for key in
                       ("schema_version", "producer_notebook", "config_hash", "upstream_manifest_hash"))
        if (row.get("artifact_path") != name or name in seen or actual != expected
                or not manifest_digest_valid(row.get("sha256"))
                or type(row.get("bytes")) is not int or row["bytes"] < 0):
            raise RuntimeError(f"Invalid Stage {stage} artifact provenance: {name}")
        seen.add(name)

import gzip
import shutil


PROMPT_CHUNKS = "data/audits/rendered_prompt_payload_chunks/"
PROMPT_INDEX = "data/audits/rendered_prompt_payload_index.jsonl"
PROMPT_MEASUREMENTS = "data/audits/rendered_prompt_measurements.jsonl"


def safe_source_file(root, path):
    return path.is_file() and not path.is_symlink() and path.resolve().is_relative_to(root.resolve())


def verified_jsonl(root, artifacts, name):
    row = artifacts.get(name)
    path = root / name
    if (row is None or not safe_source_file(root, path)
            or path.stat().st_size != row["bytes"] or file_digest(path) != row["sha256"]):
        raise RuntimeError(f"Stage 02 source ledger failed its manifest check: {name}")
    with path.open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]


def expanded_prompt_file(root, name):
    expected = root / name
    stem = expected.with_suffix("")
    candidates = {
        stem,
        stem / (expected.name + ".tmp"),
        stem / expected.name,
        stem / stem.name,
        expected.with_name(expected.name + ".tmp"),
    }
    for filename in (expected.name + ".tmp", stem.name):
        candidates.update(expected.parent.rglob(filename))
    found = sorted(path for path in candidates if safe_source_file(root, path))
    if len(found) != 1:
        raise RuntimeError(
            f"Stage 02 expected one expanded JSONL file for {name}, found {len(found)}. "
            "Supply the original ZIP with STAGE_02_ARCHIVE if the expanded file is unavailable."
        )
    return found[0]


def validate_expanded_chunk(path, expected):
    with path.open(encoding="utf-8") as stream:
        records = (json.loads(line) for line in stream if line.strip())
        for index, (payload, measured) in enumerate(zip(records, expected, strict=True), 1):
            prompt, tokens = payload["rendered_prompt"], payload["token_ids"]
            if (not isinstance(prompt, str) or not isinstance(tokens, list)
                    or any(type(token) is not int or token < 0 for token in tokens)):
                raise RuntimeError(f"Invalid prompt or token sequence in {path.name}, record {index}")
            prompt_hash = hashlib.sha256(prompt.encode("utf-8")).hexdigest()
            token_hash = provenance_digest(tokens)
            if (payload["op_id"] != measured["op_id"] or payload["encoding"] != "utf-8"
                    or payload["boundary"] != measured["boundary"]
                    or prompt_hash != payload["rendered_prompt_sha256"]
                    or prompt_hash != measured["rendered_prompt_sha256"]
                    or token_hash != payload["token_ids_sha256"]
                    or token_hash != measured["token_ids_sha256"]
                    or len(tokens) != payload["token_count"]
                    or len(tokens) != measured["rendered_prompt_tokens"]):
                raise RuntimeError(f"Stage 02 prompt verification failed in {path.name}, record {index}")


def recover_expanded_stage02(root, manifest, missing_names, recovery_root):
    """Validate exact prompt content and create a new, explicitly identified gzip representation."""
    artifacts = {artifact_path_for(row): row for row in manifest["artifacts"]}
    if len(artifacts) != manifest["artifact_count"] or len(artifacts) != len(manifest["artifacts"]):
        raise RuntimeError("Stage 02 original manifest contains duplicate or missing artifact entries.")
    index = verified_jsonl(root, artifacts, PROMPT_INDEX)
    measurements = verified_jsonl(root, artifacts, PROMPT_MEASUREMENTS)
    if (len(index) != manifest["prompt_payload_chunk_count"]
            or len(measurements) != manifest["operation_count"]
            or len({row["op_id"] for row in measurements}) != len(measurements)):
        raise RuntimeError("Stage 02 prompt ledger counts disagree with the original manifest.")
    missing = set(missing_names)
    original_records = {}
    replacements = {}
    cursor = 0
    valid_total = 0
    for number, chunk in enumerate(index):
        name = f"{PROMPT_CHUNKS}chunk_{number:05d}.jsonl.gz"
        artifact = artifacts.get(name)
        count = chunk["operation_count"]
        assigned = measurements[cursor:cursor + count]
        cursor += count
        if (artifact is None or chunk["chunk_number"] != number or count <= 0
                or len(assigned) != count
                or provenance_digest([row["op_id"] for row in assigned]) != chunk["operation_ids_sha256"]
                or chunk["payload_sha256"] != artifact["sha256"]
                or chunk["payload_bytes"] != artifact["bytes"]
                or any(row["renderer_fingerprint"] != chunk["renderer_fingerprint"] for row in assigned)):
            raise RuntimeError(f"Stage 02 chunk index disagrees with the original manifest: {name}")
        valid = [row for row in assigned if row["slice_status"] == "PASS"]
        if len(valid) != chunk["payload_record_count"]:
            raise RuntimeError(f"Stage 02 prompt count mismatch: {name}")
        valid_total += len(valid)
        if name not in missing:
            continue
        source = expanded_prompt_file(root, name)
        validate_expanded_chunk(source, valid)
        destination = recovery_root / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        with source.open("rb") as plain, destination.open("wb") as compressed:
            with gzip.GzipFile(filename="", mode="wb", fileobj=compressed, compresslevel=6, mtime=0) as stream:
                shutil.copyfileobj(plain, stream, length=1024 * 1024)
        original_records[name] = {key: artifact[key] for key in ("bytes", "sha256")}
        artifact.update(bytes=destination.stat().st_size, sha256=file_digest(destination))
        replacements[name] = destination
        if len(replacements) % 25 == 0 or len(replacements) == len(missing):
            print(f"Verified Stage 02 prompt chunks: {len(replacements)}/{len(missing)}", flush=True)
    if (cursor != len(measurements) or valid_total != manifest["valid_slice_count"]
            or set(replacements) != missing):
        raise RuntimeError("Stage 02 recovered chunks do not reconcile with the original ledgers.")
    manifest["reexport_of"]["prompt_payload_representation"] = {
        "method": "gzip_repack_from_verified_expanded_jsonl",
        "verification": "original indexed operation order, prompt SHA-256, token SHA-256, counts and boundaries",
        "original_chunk_records": original_records,
        "original_chunk_index": PROMPT_INDEX,
        "original_measurement_ledger": PROMPT_MEASUREMENTS,
    }
    manifest["artifact_bytes"] = sum(row["bytes"] for row in manifest["artifacts"])
    if "artifact_gib" in manifest:
        manifest["artifact_gib"] = round(manifest["artifact_bytes"] / (1024 ** 3), 6)
    manifest["project_tree_hash"] = provenance_digest([
        {"path": artifact_path_for(row), "bytes": row["bytes"], "sha256": row["sha256"]}
        for row in manifest["artifacts"]
    ])
    return replacements

import copy
import stat
import tempfile
import zipfile
from pathlib import Path


def file_digest(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def manifest_bytes(value):
    return (json.dumps(value, sort_keys=True, indent=2, ensure_ascii=False, allow_nan=False)
            + "\n").encode("utf-8")


def repackage_stages(input_root, output_root, *, stage02_archive=None):
    """Verify the existing Stage 00–02 release chain and publish its §17 metadata revision."""
    input_root, output_root = Path(input_root), Path(output_root)
    stages = ("00", "01", "02")
    output_root.mkdir(parents=True, exist_ok=True)
    sources = {}
    for stage in stages:
        sidecar_name = f"stage_{stage}_manifest.json"
        sidecars = sorted(input_root.rglob(sidecar_name))
        if len(sidecars) != 1:
            raise RuntimeError(f"Attach one completed Stage {stage} dataset. Found {len(sidecars)} manifests.")
        external = json.loads(sidecars[0].read_bytes())
        dataset = sidecars[0].parent
        archive_field = external["archive"]
        if isinstance(archive_field, str):
            archive_name, archive_hash = archive_field, external["archive_sha256"]
        else:
            archive_name = archive_field.get("name") or archive_field["filename"]
            archive_hash = archive_field["sha256"]
        if not isinstance(archive_name, str) or PurePosixPath(archive_name).name != archive_name:
            raise RuntimeError(f"Invalid Stage {stage} archive name")
        source_archive = dataset / archive_name
        if stage == "02":
            if stage02_archive:
                source_archive = Path(stage02_archive)
                if not source_archive.is_file():
                    raise RuntimeError(f"Stage 02 archive override is not a file: {source_archive}")
            elif not source_archive.is_file():
                candidates = sorted(path for path in input_root.rglob(archive_name + ".bin")
                                    if path.is_file())
                if len(candidates) > 1:
                    raise RuntimeError("Set STAGE_02_ARCHIVE to the exact path of one Stage 02 archive.")
                if candidates:
                    source_archive = candidates[0]
        member = f"prose_operation_experiment/registries/{stage}_manifest.json"
        if source_archive.is_file():
            if file_digest(source_archive) != archive_hash:
                raise RuntimeError(f"Stage {stage} source archive hash mismatch")
            with zipfile.ZipFile(source_archive) as archive:
                raw = archive.read(member)
            expanded_root = None
        else:
            matches = sorted(dataset.rglob(member))
            if len(matches) != 1:
                raise RuntimeError(f"Stage {stage} ZIP or one expanded project directory is required")
            raw = matches[0].read_bytes()
            expanded_root = matches[0].parent.parent
        old_digest = hashlib.sha256(raw).hexdigest()
        internal = json.loads(raw)
        if old_digest != (external.get("internal_manifest_sha256")
                          or external.get("internal_manifest", {}).get("sha256")):
            raise RuntimeError(f"Stage {stage} registry hash differs from its sidecar")
        for manifest in (internal, external):
            if manifest.get("stage") != stage or manifest.get("stage_status", manifest.get("status")) != "PASS":
                raise RuntimeError(f"Stage {stage} did not pass")
        if stage != "00" and internal.get("design_id") != "openapi-fresh-pilot-v1":
            raise RuntimeError(f"Stage {stage} is from the historical experiment")
        links = internal.get("upstream_manifest_sha256", {})
        expected = {earlier: sources[earlier]["digest"] for earlier in stages if earlier < stage}
        if links != expected:
            raise RuntimeError(f"Stage {stage} did not consume these exact original upstream manifests")
        if not manifest_digest_valid(internal.get("configuration_hash")):
            raise RuntimeError(f"Stage {stage} has no recorded configuration hash")
        names = {artifact_path_for(row) for row in internal["artifacts"]}
        if stage != "00" and "data/inventory/families.csv" not in names:
            raise RuntimeError(f"Stage {stage} lacks the reviewed family registry")
        if stage == "02" and external.get("may_begin_notebook_03") is not True:
            raise RuntimeError("Stage 02 did not authorize Notebook 03")
        missing_gzip = sorted(name for name in names
                              if stage == "02" and expanded_root is not None
                              and name.startswith(PROMPT_CHUNKS) and name.endswith(".jsonl.gz")
                              and not (expanded_root / name).is_file())
        sources[stage] = {"internal": internal, "external": external, "digest": old_digest,
                          "archive": source_archive if expanded_root is None else None,
                          "archive_hash": archive_hash, "root": expanded_root, "member": member,
                          "missing_gzip": missing_gzip}

    published = {}
    for stage, source in sources.items():
        output = output_root / f"stage_{stage}_provenance_dataset"
        revised = copy.deepcopy(source["internal"])
        original_links = revised.get("upstream_manifest_sha256", {})
        revised["reexport_of"] = {
            "internal_manifest_sha256": source["digest"],
            "upstream_manifest_sha256": original_links,
            "recorded_archive_sha256": source["archive_hash"],
            "archive_digest_verified": source["archive"] is not None,
        }
        revised["upstream_manifest_sha256"] = {key: published[key] for key in original_links}
        with tempfile.TemporaryDirectory(dir=output_root, prefix=f".stage_{stage}_") as work:
            work = Path(work)
            temp = work / "dataset"
            temp.mkdir()
            recovered = (recover_expanded_stage02(
                source["root"], revised, source["missing_gzip"], work / "recovered")
                if source["missing_gzip"] else {})
            register_artifact_provenance(revised)
            new_bytes = manifest_bytes(revised)
            new_digest = hashlib.sha256(new_bytes).hexdigest()
            archive_name = f"stage_{stage}_provenance_v1.zip"
            archive_path = temp / archive_name
            origin = zipfile.ZipFile(source["archive"]) if source["archive"] else None
            try:
                if origin:
                    members = {f"prose_operation_experiment/{artifact_path_for(row)}"
                               for row in revised["artifacts"]} | {source["member"]}
                    info = [entry for entry in origin.infolist() if not entry.is_dir()]
                    if (len(info) != len(members) or {entry.filename for entry in info} != members
                            or any(stat.S_ISLNK(entry.external_attr >> 16) for entry in info)):
                        raise RuntimeError(f"Stage {stage} ZIP inventory differs from its manifest")
                with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED, compresslevel=4) as target:
                    for row in revised["artifacts"]:
                        name = artifact_path_for(row)
                        member = f"prose_operation_experiment/{name}"
                        if origin:
                            stream = origin.open(member)
                        else:
                            path = recovered.get(name, source["root"] / name)
                            if not safe_source_file(work if name in recovered else source["root"], path):
                                raise RuntimeError(f"Missing or unsafe Stage {stage} artifact: {name}")
                            stream = path.open("rb")
                        digest, size = hashlib.sha256(), 0
                        with stream, target.open(member, "w", force_zip64=True) as destination:
                            while block := stream.read(1024 * 1024):
                                destination.write(block)
                                digest.update(block)
                                size += len(block)
                        if digest.hexdigest() != row["sha256"] or size != row["bytes"]:
                            raise RuntimeError(f"Stage {stage} artifact mismatch: {name}")
                    target.writestr(zipfile.ZipInfo(source["member"]), new_bytes,
                                    compress_type=zipfile.ZIP_DEFLATED)
                with zipfile.ZipFile(archive_path) as target:
                    corrupt = target.testzip()
                    if corrupt is not None:
                        raise RuntimeError(f"Re-exported ZIP failed its CRC check: {corrupt}")
                    member_count = len(target.infolist())
                    uncompressed_bytes = sum(entry.file_size for entry in target.infolist())
            finally:
                if origin is not None:
                    origin.close()

            external = copy.deepcopy(source["external"])
            archive_hash = file_digest(archive_path)
            if isinstance(external["archive"], dict):
                archive_info = external["archive"]
                archive_info.update(sha256=archive_hash, bytes=archive_path.stat().st_size,
                                    member_count=member_count)
                archive_info["name" if "name" in archive_info else "filename"] = archive_name
                if "uncompressed_member_bytes" in archive_info:
                    archive_info["uncompressed_member_bytes"] = uncompressed_bytes
            else:
                external.update(archive=archive_name, archive_sha256=archive_hash,
                                archive_bytes=archive_path.stat().st_size)
            external.update(internal_manifest_sha256=new_digest,
                            artifact_record_schema=ARTIFACT_SCHEMA,
                            reexport_of=revised["reexport_of"])
            if isinstance(external.get("internal_manifest"), dict):
                external["internal_manifest"].update(sha256=new_digest)
                if "bytes" in external["internal_manifest"]:
                    external["internal_manifest"]["bytes"] = len(new_bytes)
                if recovered:
                    external["internal_manifest"].update(
                        artifact_bytes=revised["artifact_bytes"], project_tree_hash=revised["project_tree_hash"])
            verification_note = (
                f"{len(recovered)} gzip chunks were repacked from the mounted JSONL. "
                "Prompt and token hashes, record order, and boundaries were verified against the original "
                "measurement ledger. Original compressed hashes and new artifact hashes are recorded.\n"
                if recovered else
                "All registered artifact bytes were copied and verified against the original manifest.\n"
            )
            readme = (f"# Stage {stage} verified provenance re-export\n\n"
                      f"Original manifest SHA-256: `{source['digest']}`\n\n"
                      + verification_note
                      + "The revised registry records the original lineage and links to the revised upstream registries.\n")
            readme_path = temp / "README.md"
            readme_path.write_text(readme, encoding="utf-8")
            if isinstance(external.get("readme"), dict):
                external["readme"].update(name="README.md", sha256=file_digest(readme_path))
            sidecar = temp / f"stage_{stage}_manifest.json"
            sidecar.write_bytes(manifest_bytes(external))
            (temp / f"stage_{stage}_checksums.sha256").write_text(
                "".join(f"{file_digest(path)}  {path.name}\n" for path in
                        (archive_path, sidecar, readme_path)), encoding="utf-8")
            if output.exists():
                if {path.name for path in output.iterdir()} != {path.name for path in temp.iterdir()} or any(
                    file_digest(output / path.name) != file_digest(path) for path in temp.iterdir()
                ):
                    raise RuntimeError(f"Existing output differs. Choose an unused output folder: {output}")
            else:
                # Rename the finished directory on the same filesystem.
                temp.rename(output)
            published[stage] = new_digest
            print(f"Stage {stage} PASS · {len(revised['artifacts']):,} artifacts · {output}")
    return published

# Attach the three completed Stage 00–02 datasets in a separate CPU notebook.
# Leave this unset to use the attached datasets. If the expanded JSONL
# is unavailable, enter the exact path of the original Stage 02 ZIP.
STAGE_02_ARCHIVE = None
repackage_stages("/kaggle/input", "/kaggle/working", stage02_archive=STAGE_02_ARCHIVE)


Stage 00 PASS · 32 artifacts · /kaggle/working/stage_00_provenance_dataset
Stage 01 PASS · 10,335 artifacts · /kaggle/working/stage_01_provenance_dataset
Verified Stage 02 prompt chunks: 25/235
Verified Stage 02 prompt chunks: 50/235
Verified Stage 02 prompt chunks: 75/235
Verified Stage 02 prompt chunks: 100/235
Verified Stage 02 prompt chunks: 125/235
Verified Stage 02 prompt chunks: 150/235
Verified Stage 02 prompt chunks: 175/235
Verified Stage 02 prompt chunks: 200/235
Verified Stage 02 prompt chunks: 225/235
Verified Stage 02 prompt chunks: 235/235
Stage 02 PASS · 58,881 artifacts · /kaggle/working/stage_02_provenance_dataset


{'00': 'a064f3bb9814b3545ed81984c5a5bcd5a5ce236ce621f2fd0d039950efafc431',
 '01': 'e6372a8f281da58f893d9b34c8516d1472f9f7cfb034d52265b1b58582897208',
 '02': '538df541290ea421a1dab61cdd3a78a072d63d4ab109f2d902b805d7e728b5fd'}